# Import modules

In [ ]:
import pandas as pd
import numpy as np
import math
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, HashingVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression, RidgeClassifier, Perceptron, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import NearestCentroid
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.metrics import confusion_matrix, classification_report, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize


# Load Dataset

In [27]:

dataset = pd.read_csv('news.csv', index_col='Unnamed: 0').reset_index(drop=True)
df_X = dataset['text']
df_y = dataset['label']
X_train, X_test, y_train, y_test = train_test_split(df_X, df_y, test_size=0.25, random_state=10)


# Vectorization setup

In [3]:
coverage = 0.95
allowed_pairs = 100
eps = 0.01

cv_for_stats = CountVectorizer(stop_words='english')
Xc = cv_for_stats.fit_transform(X_train)
freq = np.asarray(Xc.sum(axis=0)).ravel()
order = np.argsort(freq)[::-1]
cum = np.cumsum(freq[order]) / freq.sum()
K = int(np.searchsorted(cum, coverage) + 1)

K_eff = K                     # only frequent terms
eps_eff = 0.1                 # allow 10% collision probability

m_req_prob = int(K_eff * (K_eff - 1) / (-2 * math.log(1 - eps_eff)))

def next_pow2(x):
    return 1 << (x - 1).bit_length()

n_features_choice = next_pow2(m_req_prob)
n_features_choice = min(n_features_choice, 2**18)

print(f"Terms covering {coverage*100:.0f}%: {K}, Hashing n_features={n_features_choice}")


Terms covering 95%: 17904, Hashing n_features=262144


# Model Definitions

In [4]:

def get_model(model_name):
    models = {
        'LogisticRegression': LogisticRegression(),
        'LinearSVC': LinearSVC(),
        'RidgeClassifier': RidgeClassifier(),
        'Perceptron': Perceptron(),
        'NearestCentroid': NearestCentroid(),
        'SGDClassifier': SGDClassifier(),
        'MultinomialNB': MultinomialNB(),
        'BernoulliNB': BernoulliNB()
    }
    return models[model_name]


# Parameter grids


In [5]:
base_grids = {
    'LogisticRegression': [
        {'C': [0.1, 1], 'max_iter': [3000]}
    ],

    'LinearSVC': [
        {'C': [0.1, 1]}
    ],

    'RidgeClassifier': [
        {'alpha': [1.0, 10.0]}
    ],

    'Perceptron': [
        {'alpha': [1e-4, 1e-3], 'max_iter': [2000]}
    ],

    'MultinomialNB': [
        {'alpha': [0.1, 1.0]}
    ],

    'BernoulliNB': [
        {'alpha': [0.1, 1.0]}
    ],

    'SGDClassifier': [
        {'alpha': [1e-4, 1e-3], 'loss': ['log_loss'], 'max_iter': [2000]}
    ]
}

hash_grids = {
    'LogisticRegression': [
        {'C': [1], 'max_iter': [5000]}
    ],

    'LinearSVC': [
        {'C': [1]}
    ],

    'RidgeClassifier': [
        {'alpha': [1.0]}
    ],

    'Perceptron': [
        {'alpha': [1e-4], 'max_iter': [2000]}
    ],

    'SGDClassifier': [
        {'alpha': [1e-4], 'loss': ['log_loss'], 'max_iter': [2000]}
    ]
}


# Classification function


In [6]:

def run_classification_model(X_train_vec, y_train, X_test_vec, y_test, model_name, param_grid, grid_cv):
    model = get_model(model_name)
    grid = GridSearchCV(model, param_grid, scoring='f1_weighted', cv=grid_cv, n_jobs=-1, refit=True, error_score='raise')
    grid.fit(X_train_vec, y_train)
    y_pred = grid.predict(X_test_vec)
    f1 = f1_score(y_test, y_pred, average='weighted')
    cm = confusion_matrix(y_test, y_pred)
    cr = classification_report(y_test, y_pred)

    # ROC AUC
    if len(set(y_test)) > 2:
        if hasattr(grid.best_estimator_, "predict_proba"):
            roc_auc = roc_auc_score(y_test, grid.predict_proba(X_test_vec), multi_class='ovr')
        else:
            scores = grid.decision_function(X_test_vec)
            roc_auc = roc_auc_score(label_binarize(y_test, classes=np.unique(y_test)), scores, multi_class='ovr')
    else:
        if hasattr(grid.best_estimator_, "predict_proba"):
            roc_auc = roc_auc_score(y_test, grid.predict_proba(X_test_vec)[:,1])
        else:
            scores = grid.decision_function(X_test_vec)
            roc_auc = roc_auc_score(y_test, scores)

    # Output Results
    print('.' * 25)
    print(f"{model_name} \nBest Parameters: {grid.best_params_}")
    print(f"F1 Score (weighted):{f1:.4f} \nROC AUC Score:{roc_auc:.4f}")
    print("Confusion Matrix:\n", cm)
    print("Classification Report:\n", cr)
    
    
    return grid, f1, roc_auc


# Loop over vectorizers


In [15]:

vectorizers = {
    'CountVectorizer': CountVectorizer(stop_words='english', max_features=K),
    'TfidfVectorizer': TfidfVectorizer(stop_words='english', max_features=K),
    'HashingVectorizer': HashingVectorizer(stop_words='english', n_features=n_features_choice, alternate_sign=False, norm=None)
}

full_db = pd.DataFrame()
i=0

for vec_name, vect in vectorizers.items():
    print(f"\n--- Vectorizer: {vec_name} ---")

    if vec_name == 'HashingVectorizer':
        X_train_vec = vect.transform(X_train)
        X_test_vec = vect.transform(X_test)
        grid_cv = 3
        models_with_param_grids = hash_grids
    else:
        X_train_vec = vect.fit_transform(X_train)
        X_test_vec = vect.transform(X_test)
        grid_cv = 5
        models_with_param_grids = base_grids

    vectorizer_results = {}

    for model_name, param_grid in models_with_param_grids.items():
        grid, f1, roc_auc = run_classification_model(
            X_train_vec, y_train,
            X_test_vec, y_test,
            model_name, param_grid, grid_cv
        )
        i+=1
        vectorizer_results[i] = {
            'vectorizer_name': vec_name,
            'model_name': model_name,
            'f1_score': f1,
            'roc_auc': roc_auc,
            'best_params': grid.best_params_,
            'model': grid,
            'vectorizer': vect
        }

    sorted_df = (
        pd.DataFrame.from_dict(vectorizer_results, orient='index')
        .sort_values(by=['roc_auc', 'f1_score'], ascending=False)
    )

    print('*'*50)
    print(f"Top model for {vec_name}: {sorted_df.iloc[0].name} | "
          f"F1={sorted_df.iloc[0]['f1_score']:.4f} | "
          f"ROC AUC={sorted_df.iloc[0]['roc_auc']:.4f}")
    print('*'*50)

    full_db = pd.concat([full_db, sorted_df])

   



--- Vectorizer: CountVectorizer ---
.........................
LogisticRegression 
Best Parameters: {'C': 0.1, 'max_iter': 3000}
F1 Score (weighted):0.9293 
ROC AUC Score:0.9750
Confusion Matrix:
 [[726  43]
 [ 69 746]]
Classification Report:
               precision    recall  f1-score   support

        FAKE       0.91      0.94      0.93       769
        REAL       0.95      0.92      0.93       815

    accuracy                           0.93      1584
   macro avg       0.93      0.93      0.93      1584
weighted avg       0.93      0.93      0.93      1584



d:\HopeAI\.venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


.........................
LinearSVC 
Best Parameters: {'C': 0.1}
F1 Score (weighted):0.9085 
ROC AUC Score:0.9585
Confusion Matrix:
 [[710  59]
 [ 86 729]]
Classification Report:
               precision    recall  f1-score   support

        FAKE       0.89      0.92      0.91       769
        REAL       0.93      0.89      0.91       815

    accuracy                           0.91      1584
   macro avg       0.91      0.91      0.91      1584
weighted avg       0.91      0.91      0.91      1584

.........................
RidgeClassifier 
Best Parameters: {'alpha': 10.0}
F1 Score (weighted):0.8241 
ROC AUC Score:0.8975
Confusion Matrix:
 [[682  87]
 [191 624]]
Classification Report:
               precision    recall  f1-score   support

        FAKE       0.78      0.89      0.83       769
        REAL       0.88      0.77      0.82       815

    accuracy                           0.82      1584
   macro avg       0.83      0.83      0.82      1584
weighted avg       0.83      0

d:\HopeAI\.venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


.........................
LinearSVC 
Best Parameters: {'C': 1}
F1 Score (weighted):0.8902 
ROC AUC Score:0.9431
Confusion Matrix:
 [[690  79]
 [ 95 720]]
Classification Report:
               precision    recall  f1-score   support

        FAKE       0.88      0.90      0.89       769
        REAL       0.90      0.88      0.89       815

    accuracy                           0.89      1584
   macro avg       0.89      0.89      0.89      1584
weighted avg       0.89      0.89      0.89      1584

.........................
RidgeClassifier 
Best Parameters: {'alpha': 1.0}
F1 Score (weighted):0.7783 
ROC AUC Score:0.8329
Confusion Matrix:
 [[652 117]
 [233 582]]
Classification Report:
               precision    recall  f1-score   support

        FAKE       0.74      0.85      0.79       769
        REAL       0.83      0.71      0.77       815

    accuracy                           0.78      1584
   macro avg       0.78      0.78      0.78      1584
weighted avg       0.79      0.78

In [20]:
full_db.sort_values(by=['f1_score', 'roc_auc'], ascending=False)


,vectorizer_name,model_name,f1_score,roc_auc,best_params,model,vectorizer
9,TfidfVectorizer,LinearSVC,0.941931,0.984387,{'C': 1},"GridSearchCV(cv=5, error_score='raise', estima...","TfidfVectorizer(max_features=17904, stop_words..."
10,TfidfVectorizer,RidgeClassifier,0.940668,0.983890,{'alpha': 1.0},"GridSearchCV(cv=5, error_score='raise', estima...","TfidfVectorizer(max_features=17904, stop_words..."
14,TfidfVectorizer,SGDClassifier,0.931830,0.979896,"{'alpha': 0.0001, 'loss': 'log_loss', 'max_ite...","GridSearchCV(cv=5, error_score='raise', estima...","TfidfVectorizer(max_features=17904, stop_words..."
1,CountVectorizer,LogisticRegression,0.929308,0.975025,"{'C': 0.1, 'max_iter': 3000}","GridSearchCV(cv=5, error_score='raise', estima...","CountVectorizer(max_features=17904, stop_words..."
7,CountVectorizer,SGDClassifier,0.928671,0.970179,"{'alpha': 0.001, 'loss': 'log_loss', 'max_iter...","GridSearchCV(cv=5, error_score='raise', estima...","CountVectorizer(max_features=17904, stop_words..."
15,HashingVectorizer,LogisticRegression,0.924258,0.969464,"{'C': 1, 'max_iter': 5000}","GridSearchCV(cv=3, error_score='raise', estima...","HashingVectorizer(alternate_sign=False, n_feat..."
11,TfidfVectorizer,Perceptron,0.922352,0.976003,"{'alpha': 0.0001, 'max_iter': 2000}","GridSearchCV(cv=5, error_score='raise', estima...","TfidfVectorizer(max_features=17904, stop_words..."
8,TfidfVectorizer,LogisticRegression,0.921099,0.975321,"{'C': 1, 'max_iter': 3000}","GridSearchCV(cv=5, error_score='raise', estima...","TfidfVectorizer(max_features=17904, stop_words..."
19,HashingVectorizer,SGDClassifier,0.920996,0.961350,"{'alpha': 0.0001, 'loss': 'log_loss', 'max_ite...","GridSearchCV(cv=3, error_score='raise', estima...","HashingVectorizer(alternate_sign=False, n_feat..."
4,CountVectorizer,Perceptron,0.919104,0.966412,"{'alpha': 0.0001, 'max_iter': 2000}","GridSearchCV(cv=5, error_score='raise', estima...","CountVectorizer(max_features=17904, stop_words..."


In [22]:
best_model = full_db.sort_values(by=['f1_score', 'roc_auc'], ascending=False).iloc[0]
best_model


vectorizer_name                                      TfidfVectorizer
model_name                                                 LinearSVC
f1_score                                                    0.941931
roc_auc                                                     0.984387
best_params                                                 {'C': 1}
model              GridSearchCV(cv=5, error_score='raise', estima...
vectorizer         TfidfVectorizer(max_features=17904, stop_words...
Name: 9, dtype: object

# Test with sample

In [33]:
print(X_test[[1716]],'\n', y_test[1716])


1716    The Latest On Paris Attack: Manhunt Continues;...
Name: text, dtype: object 
 REAL


In [39]:
single_news = X_test[[1716]]  

vect = best_model['vectorizer']
model = best_model['model']
single_vec = vect.transform(single_news)
print(model.predict(single_vec))


['REAL']
